```
flow
1. start (read csv)
2. cleaning (validation, normalisasi, filtering)
3. classifier --> indobert
4. agregat per toko (scoring, ekstraksi problem?)
5. end (to csv)
```

### install & load library

In [ ]:
!pip install -q transformers torch pandas tqdm

import pandas as pd
import numpy as np
import torch
import re
from transformers import pipeline
from tqdm.auto import tqdm
from collections import Counter
from google.colab import drive
drive.mount('/content/drive')
import json

Mounted at /content/drive


In [ ]:
!pip install -q keybert

from keybert import KeyBERT

kw_model = KeyBERT(model='indobenchmark/indobert-base-p2')

In [ ]:
pip install bertopic sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.5 MB/s eta 0:00:00


In [ ]:
# file lama
file_path = '/content/drive/MyDrive/sentiment0_290126.json'

with open(file_path, 'r') as f:
    sentimen_lama = json.load(f)

In [ ]:
df_sentimen_lama = pd.read_json(file_path)
df_sentimen_lama.head()

,place_name,place_lat,place_lng,sector,reviewer_name,rating,time,text,text_clean,sentiment,score
0,Bebek Mas Mun BakauHeni,-5.865811,105.756932,lahan belum dikuasai,Catherine Puspita,5,2 tahun lalu,Bebek enak dan empuk. Bisa dimakan dengan muda...,bebek enak dan empuk bisa dimakan dengan muda...,Positif,0.999970
1,Bebek Mas Mun BakauHeni,-5.865811,105.756932,lahan belum dikuasai,Ernie Arbie,5,sebulan lalu,Favorit bgt bebek mas mun ... Suka bebeknya ya...,favorit banget bebek mas mun suka bebeknya...,Positif,0.999969
2,Bebek Mas Mun BakauHeni,-5.865811,105.756932,lahan belum dikuasai,Mahfut Arifin,5,2 bulan lalu,Makan bebek d sini enak bangett rekomen bebekn...,makan bebek d sini enak bangett bagus bebekny ...,Positif,0.999970
3,Bebek Mas Mun BakauHeni,-5.865811,105.756932,lahan belum dikuasai,BeWee,5,3 tahun lalu,Mantap ada lalapan ayam terjangkau. Ada harga ...,bagus ada lalapan ayam terjangkau ada harga n...,Positif,0.999957
4,Bebek Mas Mun BakauHeni,-5.865811,105.756932,lahan belum dikuasai,Benny Manurung,1,6 bulan lalu,Karyawan nya gak niat jualan....1 jam lebih nu...,karyawan nya tidak niat jualan 1 jam lebih ...,Negatif,0.999872


In [ ]:
# file baru
path_file = '/content/drive/MyDrive/sentimen_zonabaru.csv'
df_sentimen_baru = pd.read_csv(path_file)

df_sentimen_baru

,place_name,sector,reviewer_name,rating,time,text,text_clean,sentiment,score,Zonasi Kawasan
0,Bebek Mas Mun BakauHeni,lahan belum dikuasai,Catherine Puspita,5,2 tahun lalu,Bebek enak dan empuk. Bisa dimakan dengan muda...,bebek enak dan empuk bisa dimakan dengan muda...,Positif,0.999970,Harbourfront and Intermoda Area
1,Bebek Mas Mun BakauHeni,lahan belum dikuasai,Ernie Arbie,5,sebulan lalu,Favorit bgt bebek mas mun ... Suka bebeknya ya...,favorit banget bebek mas mun suka bebeknya...,Positif,0.999969,Harbourfront and Intermoda Area
2,Bebek Mas Mun BakauHeni,lahan belum dikuasai,Mahfut Arifin,5,2 bulan lalu,Makan bebek d sini enak bangett rekomen bebekn...,makan bebek d sini enak bangett bagus bebekny ...,Positif,0.999970,Harbourfront and Intermoda Area
3,Bebek Mas Mun BakauHeni,lahan belum dikuasai,BeWee,5,3 tahun lalu,Mantap ada lalapan ayam terjangkau. Ada harga ...,bagus ada lalapan ayam terjangkau ada harga n...,Positif,0.999957,Harbourfront and Intermoda Area
4,Bebek Mas Mun BakauHeni,lahan belum dikuasai,Benny Manurung,1,6 bulan lalu,Karyawan nya gak niat jualan....1 jam lebih nu...,karyawan nya tidak niat jualan 1 jam lebih ...,Negatif,0.999872,Harbourfront and Intermoda Area
...,...,...,...,...,...,...,...,...,...,...
276,Alfamart lintas bakauheni,lahan belum dikuasai,Muhamad Jazuli,5,5 bulan lalu,"Pegawai nya ramah, tempat nya bersih",pegawai nya ramah tempat nya bersih,Positif,0.999964,di luar area
277,Alfamart lintas bakauheni,lahan belum dikuasai,Luqman,5,2 tahun lalu,اَلْحَمْدُلِلّهِ\nBeli kebutuhan untuk naik kapal,ا ل ح م د ل ل ه \nbeli kebutuhan untuk naik kapal,Netral,0.999888,di luar area
278,Alfamart lintas bakauheni,lahan belum dikuasai,Asram Siregar,5,2 tahun lalu,"24hours, bebas parkir mantap...\nPerhatikan ke...",24hours bebas parkir bagus \nperhatikan keb...,Positif,0.999853,di luar area
279,Alfamart lintas bakauheni,lahan belum dikuasai,Rudi Exist,5,2 tahun lalu,Dengan operasi 24 jam membatu orang yang henda...,dengan operasi 24 jam membatu orang yang henda...,Positif,0.999955,di luar area


In [ ]:
# merge data baru ke data lama
df_coords = df_sentimen_lama[['text', 'place_lat', 'place_lng']]

# 'on=text' artinya mencari teks ulasan yang sama di kedua file
df_final = pd.merge(df_sentimen_baru, df_coords, on='text', how='left')

# Ganti isi 'sector' lama dengan 'Zonasi Kawasan'
df_final['sector'] = df_final['Zonasi Kawasan']
df_final = df_final.drop(columns=['Zonasi Kawasan'])
df_final

,place_name,sector,reviewer_name,rating,time,text,text_clean,sentiment,score,place_lat,place_lng
0,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Catherine Puspita,5,2 tahun lalu,Bebek enak dan empuk. Bisa dimakan dengan muda...,bebek enak dan empuk bisa dimakan dengan muda...,Positif,0.999970,-5.865811,105.756932
1,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Ernie Arbie,5,sebulan lalu,Favorit bgt bebek mas mun ... Suka bebeknya ya...,favorit banget bebek mas mun suka bebeknya...,Positif,0.999969,-5.865811,105.756932
2,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Mahfut Arifin,5,2 bulan lalu,Makan bebek d sini enak bangett rekomen bebekn...,makan bebek d sini enak bangett bagus bebekny ...,Positif,0.999970,-5.865811,105.756932
3,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,BeWee,5,3 tahun lalu,Mantap ada lalapan ayam terjangkau. Ada harga ...,bagus ada lalapan ayam terjangkau ada harga n...,Positif,0.999957,-5.865811,105.756932
4,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Benny Manurung,1,6 bulan lalu,Karyawan nya gak niat jualan....1 jam lebih nu...,karyawan nya tidak niat jualan 1 jam lebih ...,Negatif,0.999872,-5.865811,105.756932
...,...,...,...,...,...,...,...,...,...,...,...
282,Alfamart lintas bakauheni,di luar area,Muhamad Jazuli,5,5 bulan lalu,"Pegawai nya ramah, tempat nya bersih",pegawai nya ramah tempat nya bersih,Positif,0.999964,-5.858546,105.743721
283,Alfamart lintas bakauheni,di luar area,Luqman,5,2 tahun lalu,اَلْحَمْدُلِلّهِ\nBeli kebutuhan untuk naik kapal,ا ل ح م د ل ل ه \nbeli kebutuhan untuk naik kapal,Netral,0.999888,-5.858546,105.743721
284,Alfamart lintas bakauheni,di luar area,Asram Siregar,5,2 tahun lalu,"24hours, bebas parkir mantap...\nPerhatikan ke...",24hours bebas parkir bagus \nperhatikan keb...,Positif,0.999853,-5.858546,105.743721
285,Alfamart lintas bakauheni,di luar area,Rudi Exist,5,2 tahun lalu,Dengan operasi 24 jam membatu orang yang henda...,dengan operasi 24 jam membatu orang yang henda...,Positif,0.999955,-5.858546,105.743721


In [ ]:
# cleaning data di luar area
df_clean = df_final[df_final['sector'] != 'di luar area'].copy()
df_clean = df_clean.dropna(subset=['sector'])
df_clean.to_csv('sentiment_final_clean.csv', index=False)

print(f"Data Berhasil Dibersihkan!")
print(f"Jumlah data awal: {len(df_final)}")
print(f"Jumlah data setelah drop: {len(df_clean)}")
print(f"Sektor yang tersedia sekarang: {df_clean['sector'].unique()}")

display(df_clean.head())

Data Berhasil Dibersihkan!
Jumlah data awal: 287
Jumlah data setelah drop: 247
Sektor yang tersedia sekarang: ['Harbourfront and Intermoda Area' 'Hilltop Resort' 'Marina District']


,place_name,sector,reviewer_name,rating,time,text,text_clean,sentiment,score,place_lat,place_lng
0,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Catherine Puspita,5,2 tahun lalu,Bebek enak dan empuk. Bisa dimakan dengan muda...,bebek enak dan empuk bisa dimakan dengan muda...,Positif,0.999970,-5.865811,105.756932
1,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Ernie Arbie,5,sebulan lalu,Favorit bgt bebek mas mun ... Suka bebeknya ya...,favorit banget bebek mas mun suka bebeknya...,Positif,0.999969,-5.865811,105.756932
2,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Mahfut Arifin,5,2 bulan lalu,Makan bebek d sini enak bangett rekomen bebekn...,makan bebek d sini enak bangett bagus bebekny ...,Positif,0.999970,-5.865811,105.756932
3,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,BeWee,5,3 tahun lalu,Mantap ada lalapan ayam terjangkau. Ada harga ...,bagus ada lalapan ayam terjangkau ada harga n...,Positif,0.999957,-5.865811,105.756932
4,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Benny Manurung,1,6 bulan lalu,Karyawan nya gak niat jualan....1 jam lebih nu...,karyawan nya tidak niat jualan 1 jam lebih ...,Negatif,0.999872,-5.865811,105.756932


In [ ]:
df = df_clean.copy()
df = df[
    ~((df['rating'] >= 3.5) & (df['sentiment'] == 'Negatif')) &
    ~((df['rating'] < 3.5) & (df['sentiment'] == 'Positif'))
]

In [ ]:
df.head()

,place_name,sector,reviewer_name,rating,time,text,text_clean,sentiment,score,place_lat,place_lng
0,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Catherine Puspita,5,2 tahun lalu,Bebek enak dan empuk. Bisa dimakan dengan muda...,bebek enak dan empuk bisa dimakan dengan muda...,Positif,0.999970,-5.865811,105.756932
1,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Ernie Arbie,5,sebulan lalu,Favorit bgt bebek mas mun ... Suka bebeknya ya...,favorit banget bebek mas mun suka bebeknya...,Positif,0.999969,-5.865811,105.756932
2,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Mahfut Arifin,5,2 bulan lalu,Makan bebek d sini enak bangett rekomen bebekn...,makan bebek d sini enak bangett bagus bebekny ...,Positif,0.999970,-5.865811,105.756932
3,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,BeWee,5,3 tahun lalu,Mantap ada lalapan ayam terjangkau. Ada harga ...,bagus ada lalapan ayam terjangkau ada harga n...,Positif,0.999957,-5.865811,105.756932
4,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Benny Manurung,1,6 bulan lalu,Karyawan nya gak niat jualan....1 jam lebih nu...,karyawan nya tidak niat jualan 1 jam lebih ...,Negatif,0.999872,-5.865811,105.756932


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 227 entries, 0 to 246
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   place_name     227 non-null    object 
 1   sector         227 non-null    object 
 2   reviewer_name  227 non-null    object 
 3   rating         227 non-null    int64  
 4   time           227 non-null    object 
 5   text           227 non-null    object 
 6   text_clean     227 non-null    object 
 7   sentiment      227 non-null    object 
 8   score          227 non-null    float64
 9   place_lat      227 non-null    float64
 10  place_lng      227 non-null    float64
dtypes: float64(3), int64(1), object(7)
memory usage: 21.3+ KB


In [ ]:
empty_rows = df[df['text_clean'].str.strip() == ""].shape[0]
print(f"jumlah baris string kosong: {empty_rows}")

jumlah baris string kosong: 31


In [ ]:
df['text_clean'] = df['text_clean'].replace(r'^\s*$', np.nan, regex=True)
df = df.dropna(subset=['text_clean'])
df = df.reset_index(drop=True)
print(f"jumlah baris setelah dibersihkan: {len(df)}")

jumlah baris setelah dibersihkan: 281


### sentiment analyst

In [ ]:
# load model
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("text-classification",
                      model="crypter70/IndoBERT-Sentiment-Analysis",
                      device=device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# proses df
texts = df['text_clean'].tolist()
results = []

for out in tqdm(classifier(texts, batch_size=32), total=len(texts)):
    results.append(out)

# mapping label
mapping_label = {"NEUTRAL": "Netral", "POSITIVE": "Positif", "NEGATIVE": "Negatif"}
df['sentiment'] = [mapping_label.get(x['label'].upper(), x['label'].upper()) for x in results]
df['score'] = [x['score'] for x in results]

  0%|          | 0/281 [00:00<?, ?it/s]

In [ ]:
df

,place_name,sector,reviewer_name,rating,time,text,text_clean,sentiment,score,place_lat,place_lng
0,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Catherine Puspita,5,2 tahun lalu,Bebek enak dan empuk. Bisa dimakan dengan muda...,bebek enak dan empuk bisa dimakan dengan muda...,Positif,0.999970,-5.865811,105.756932
1,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Ernie Arbie,5,sebulan lalu,Favorit bgt bebek mas mun ... Suka bebeknya ya...,favorit banget bebek mas mun suka bebeknya...,Positif,0.999969,-5.865811,105.756932
2,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Mahfut Arifin,5,2 bulan lalu,Makan bebek d sini enak bangett rekomen bebekn...,makan bebek d sini enak bangett bagus bebekny ...,Positif,0.999970,-5.865811,105.756932
3,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,BeWee,5,3 tahun lalu,Mantap ada lalapan ayam terjangkau. Ada harga ...,bagus ada lalapan ayam terjangkau ada harga n...,Positif,0.999957,-5.865811,105.756932
4,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Benny Manurung,1,6 bulan lalu,Karyawan nya gak niat jualan....1 jam lebih nu...,karyawan nya tidak niat jualan 1 jam lebih ...,Negatif,0.999872,-5.865811,105.756932
...,...,...,...,...,...,...,...,...,...,...,...
242,SRC TOKO AYU BAKAUHENI,Marina District,Asnawi Walet,5,9 bulan lalu,Semoga sukses,semoga sukses,Positif,0.999944,-5.867434,105.745075
243,SRC TOKO AYU BAKAUHENI,Marina District,Asnawi Walet,5,9 bulan lalu,Semoga sukses,semoga sukses,Positif,0.999944,-5.867434,105.745075
244,SRC TOKO AYU BAKAUHENI,Marina District,asnawi ilyas,5,setahun lalu,Semoga sukses,semoga sukses,Positif,0.999944,-5.867434,105.745075
245,SRC TOKO AYU BAKAUHENI,Marina District,asnawi ilyas,5,setahun lalu,Semoga sukses,semoga sukses,Positif,0.999944,-5.867434,105.745075


In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
Positif,170
Netral,31
Negatif,26


In [ ]:
df.to_json('1final_sentiment.json', orient='records', indent=4, force_ascii=False)

print("File JSON sudah siap!")

File JSON sudah siap!


## extract topic

In [ ]:
stop_words_id = [
    # --- Kata Umum Bahasa Indonesia ---
    'yang', 'dan', 'di', 'ke', 'dari', 'pada', 'dalam', 'dengan', 'untuk', 'adalah',
    'itu', 'ini', 'saya', 'kami', 'kita', 'anda', 'dia', 'mereka', 'apa', 'mana',
    'siapa', 'kapan', 'mengapa', 'bagaimana', 'bisa', 'boleh', 'ada', 'tidak', 'tak',
    'jangan', 'sudah', 'telah', 'sedang', 'akan', 'juga', 'saja', 'hanya', 'cuma',
    'tetapi', 'tapi', 'namun', 'karena', 'kalau', 'jika', 'seperti', 'bagi', 'serta',

    # --- Kata Keterangan & Penanda ---
    'sangat', 'sekali', 'banget', 'bgt', 'amat', 'terlalu', 'begitu', 'cukup',
    'lagi', 'tadi', 'nanti', 'besok', 'kemarin', 'sekarang', 'skrg', 'dulu',
    'baru', 'lama', 'bentar', 'sebentar', 'jam', 'menit', 'hari', 'bulan', 'tahun',

    # --- Singkatan & Slang (Banyak muncul di review) ---
    'yg', 'dg', 'dgn', 'klo', 'kl', 'kalo', 'udh', 'sudah', 'sdh', 'udah', 'gak',
    'ga', 'gk', 'aja', 'aj', 'jd', 'jadi', 'krn', 'karna', 'utk', 'untuk', 'pas',
    'nih', 'tuh', 'loh', 'kok', 'deh', 'lah', 'sih', 'kan', 'dong', 'yah', 'ya',
    'd', 'ke', 'pd', 'sama', 'ama', 'biar', 'buat', 'biarpun', 'dr', 'drpd',

    # --- Kata Sifat Umum (Sering jadi noise di ekstraksi topik) ---
    'bagus', 'mantap', 'mantab', 'oke', 'ok', 'mantul', 'keren', 'enak', 'favorit',
    'nyaman', 'bersih', 'ramah', 'cepat', 'lancar', 'murah', 'mahal', 'lumayan',
    'lumayanlah', 'best', 'semoga', 'sukses', 'lancar', 'jaya', 'sip',

    # --- Nama Tempat & Entitas Konteks (Opsional, agar topik lebih spesifik) ---
    'bakauheni', 'merak', 'lampung', 'pelabuhan', 'dermaga', 'kapal', 'ferry',
    'terminal', 'eksekutif', 'executive', 'mas', 'mun', 'toko', 'ayu', 'rm',
    'daerah', 'tempat', 'lokasi', 'area', 'sektor', 'district'
]

In [ ]:
def extract_topics(text):
    if not text or len(str(text)) < 5:
        return ""

    keywords = kw_model.extract_keywords(
        str(text),
        keyphrase_ngram_range=(1, 3),
        stop_words=stop_words_id,
        top_n=3
    )
    return ", ".join([k[0] for k in keywords])

df['topic'] = df['text_clean'].apply(extract_topics)
display(df[['text_clean', 'topic']].head())

,text_clean,topic
0,bebek enak dan empuk bisa dimakan dengan muda...,"bebek empuk dimakan, nya telur dadarnya, telur..."
1,favorit banget bebek mas mun suka bebeknya...,"bebeknya lembut gurih, suka bebeknya lembut, p..."
2,makan bebek d sini enak bangett bagus bebekny ...,"bumbuny meresap bgtt, bangett bebekny empuk, b..."
3,bagus ada lalapan ayam terjangkau ada harga n...,"goreng panas sambel, ayam terjangkau harga, ay..."
4,karyawan nya tidak niat jualan 1 jam lebih ...,"jualan lebih nunggu, karyawan nya niat, nya ni..."


## agregat lokasi

In [ ]:
df

,place_name,sector,reviewer_name,rating,time,text,text_clean,sentiment,score,place_lat,place_lng,topic
0,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Catherine Puspita,5,2 tahun lalu,Bebek enak dan empuk. Bisa dimakan dengan muda...,bebek enak dan empuk bisa dimakan dengan muda...,Positif,0.999970,-5.865811,105.756932,"bebek empuk dimakan, nya telur dadarnya, telur..."
1,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Ernie Arbie,5,sebulan lalu,Favorit bgt bebek mas mun ... Suka bebeknya ya...,favorit banget bebek mas mun suka bebeknya...,Positif,0.999969,-5.865811,105.756932,"bebeknya lembut gurih, suka bebeknya lembut, p..."
2,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Mahfut Arifin,5,2 bulan lalu,Makan bebek d sini enak bangett rekomen bebekn...,makan bebek d sini enak bangett bagus bebekny ...,Positif,0.999970,-5.865811,105.756932,"bumbuny meresap bgtt, bangett bebekny empuk, b..."
3,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,BeWee,5,3 tahun lalu,Mantap ada lalapan ayam terjangkau. Ada harga ...,bagus ada lalapan ayam terjangkau ada harga n...,Positif,0.999957,-5.865811,105.756932,"goreng panas sambel, ayam terjangkau harga, ay..."
4,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,Benny Manurung,1,6 bulan lalu,Karyawan nya gak niat jualan....1 jam lebih nu...,karyawan nya tidak niat jualan 1 jam lebih ...,Negatif,0.999872,-5.865811,105.756932,"jualan lebih nunggu, karyawan nya niat, nya ni..."
...,...,...,...,...,...,...,...,...,...,...,...,...
242,SRC TOKO AYU BAKAUHENI,Marina District,Asnawi Walet,5,9 bulan lalu,Semoga sukses,semoga sukses,Positif,0.999944,-5.867434,105.745075,
243,SRC TOKO AYU BAKAUHENI,Marina District,Asnawi Walet,5,9 bulan lalu,Semoga sukses,semoga sukses,Positif,0.999944,-5.867434,105.745075,
244,SRC TOKO AYU BAKAUHENI,Marina District,asnawi ilyas,5,setahun lalu,Semoga sukses,semoga sukses,Positif,0.999944,-5.867434,105.745075,
245,SRC TOKO AYU BAKAUHENI,Marina District,asnawi ilyas,5,setahun lalu,Semoga sukses,semoga sukses,Positif,0.999944,-5.867434,105.745075,


In [ ]:
import pandas as pd

# 1. Load data
df_raw = df.copy()

# 2. Agregasi per Lokasi dan Sektor
# Kita masukkan 'sector' ke dalam index agar tetap muncul di hasil akhir
df_lokasi_sektor = df_raw.pivot_table(
    index=['place_name', 'sector'],
    columns='sentiment',
    aggfunc='size',
    fill_value=0
).reset_index()

# 3. Hitung Total Ulasan per lokasi
df_lokasi_sektor['total_ulasan'] = (
    df_lokasi_sektor['Positif'] +
    df_lokasi_sektor['Negatif'] +
    df_lokasi_sektor['Netral']
)

# 4. Urutkan berdasarkan Sektor dan jumlah ulasan terbanyak
df_lokasi_sektor = df_lokasi_sektor.sort_values(by=['sector', 'total_ulasan'], ascending=[True, False])

# Tampilkan hasil
print("--- AGREGASI SENTIMEN PER LOKASI & SEKTOR ---")
display(df_lokasi_sektor)

# 5. Simpan ke JSON
df_lokasi_sektor.to_json("1final_aggregate_loc.json", orient='records', indent=4)

--- AGREGASI SENTIMEN PER LOKASI & SEKTOR ---


sentiment,place_name,sector,Negatif,Netral,Positif,total_ulasan
4,Anjungan Agung Mall,Harbourfront and Intermoda Area,1,0,9,10
15,Dermaga 5 Pelabuhan Bakauheni,Harbourfront and Intermoda Area,2,2,6,10
22,Masjid Pelabuhan Bakauheni,Harbourfront and Intermoda Area,0,2,8,10
27,Pelabuhan Bakauheni Lampung,Harbourfront and Intermoda Area,4,1,5,10
31,Terminal Executive Bakauheni,Harbourfront and Intermoda Area,2,0,8,10
13,Dermaga 1 Ferry Pelabuhan Bakauheni,Harbourfront and Intermoda Area,0,1,8,9
14,Dermaga 3 Pelabuhan Bakauheni,Harbourfront and Intermoda Area,0,3,6,9
6,Bakauheni Shopping Mall,Harbourfront and Intermoda Area,1,0,7,8
7,Bebek Mas Mun BakauHeni,Harbourfront and Intermoda Area,1,0,7,8
11,DERMAGA 2 Pelabuhan Bakauheni,Harbourfront and Intermoda Area,0,3,5,8


In [ ]:
# 4. Export ke JSON yang lebih rapi
df_final.to_json("final_summary.json", orient='records', indent=4)

In [ ]:
# 4. Export ke JSON yang lebih rapi
df_final_clean.to_json("data_analisis_final_clean.json", orient='records', indent=4)

## count

In [ ]:
# count word from topic

from collections import Counter
import pandas as pd

def get_top_topics_with_counts(series, n=5):
    # Gabungkan semua topik, pecah berdasarkan koma, dan bersihkan spasi
    all_text = ", ".join(series.astype(str))
    phrases = [p.strip() for p in all_text.split(',') if len(p.strip()) > 2]

    counts = Counter(phrases).most_common(n)

    return ", ".join([f"{topic} ({count})" for topic, count in counts])

df_agregasi = df.groupby(['sector', 'sentiment']).agg(
    jumlah_ulasan=('text_clean', 'count'),
    topik_dan_jumlah=('topic', lambda x: get_top_topics_with_counts(x, 5))
).reset_index()

df_all_sector = df.groupby('sentiment').agg(
    jumlah_ulasan=('text_clean', 'count'),
    topik_dan_jumlah=('topic', lambda x: get_top_topics_with_counts(x, 5))
).reset_index()
df_all_sector['sector'] = 'ALL SECTOR'


df_final = pd.concat([df_agregasi, df_all_sector], ignore_index=True)

df_final = df_final.sort_values(['sector', 'sentiment'], ascending=[True, False])

print("--- TABEL AGREGASI SEKTOR, SENTIMEN, & TOPIK ---")
display(df_final)

--- TABEL AGREGASI SEKTOR, SENTIMEN, & TOPIK ---


,sector,sentiment,jumlah_ulasan,topik_dan_jumlah
9,ALL SECTOR,Positif,170,"membantu (5), tempatnya luas (3), pelayananany..."
8,ALL SECTOR,Netral,31,"pulkam nugas kuliah (1), nugas kuliah disini (..."
7,ALL SECTOR,Negatif,26,"pelayanannya minus tolong (4), reguler pelayan..."
2,Harbourfront and Intermoda Area,Positif,105,"membantu (5), tempatnya luas (3), pelayananany..."
1,Harbourfront and Intermoda Area,Netral,23,"pulkam nugas kuliah (1), nugas kuliah disini (..."
0,Harbourfront and Intermoda Area,Negatif,21,"pelayanannya minus tolong (4), reguler pelayan..."
5,Hilltop Resort,Positif,53,"parkir jalan utama (1), mobil parkir jalan (1)..."
4,Hilltop Resort,Netral,8,"membeli berbagai kebutuhan (1), perjalanan mem..."
3,Hilltop Resort,Negatif,5,"ramai perbekalan penyebrangan (1), berdatangan..."
6,Marina District,Positif,12,"gurme bakar nya (1), ayam bakar bekakak (1), b..."


In [ ]:
import json

# 1. Load data dari file data_per_lahan.json
with open('data_per_lahan.json', 'r') as f:
    data_input = json.load(f)

final_aggregated = []

for sector, sentiments in data_input.items():
    # Inisialisasi pengumpul data untuk level Sektor
    combined_topics_list = []
    combined_recs_list = []
    total_ulasan = 0
    sentiment_details = []

    for item in sentiments:
        total_ulasan += item['jumlah_ulasan']

        # Ambil topik dan rekomendasi untuk digabung ke luar
        combined_topics_list.append(f"({item['sentiment']}): {item['topik_dan_jumlah']}")
        combined_recs_list.append(f"SENTIMEN {item['sentiment'].upper()}:\n{item['rekomendasi_ai']}")

        # Simpan detail teknis tetap di dalam sentimen sesuai permintaan
        sentiment_details.append({
            "sentiment": item['sentiment'],
            "jumlah_ulasan": item['jumlah_ulasan'],
            "top_topics": item['top_topics'],
            "topic_counts": item['topic_counts']
        })

    # 2. Susun struktur Flat sesuai permintaan
    entry = {
        "sector": sector,
        "total_ulasan": total_ulasan,
        # Topik dan Rekomendasi ditarik ke level Sektor (di luar sentimen)
        "all_topik_dan_jumlah": " | ".join(combined_topics_list),
        "rekomendasi_komprehensif": "\n\n---\n\n".join(combined_recs_list),
        # Detail teknis tetap di dalam
        "sentiment_breakdown": sentiment_details
    }

    final_aggregated.append(entry)

# 3. Simpan dan Download
with open('data_final_agregat_sektor.json', 'w') as f:
    json.dump(final_aggregated, f, indent=4)

from google.colab import files
files.download('data_final_agregat_sektor.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## split string ke topik dan jumlah

In [ ]:
# split data dari topik dan jumlah ke top_topics and topic_counts

def split_topic_data(val):
    if pd.isna(val):
        return None, None

    topics = [t.split('(')[0].strip() for t in val.split(',')]
    # Mencari semua angka di dalam kurung
    counts = [c.split('(')[1].replace(')', '').strip() for c in val.split(',')]

    return ", ".join(topics), ", ".join(counts)

df_final[['top_topics', 'topic_counts']] = df_final['topik_dan_jumlah'].apply(
    lambda x: pd.Series(split_topic_data(x))
)

cols = ['sector', 'sentiment', 'jumlah_ulasan', 'topik_dan_jumlah', 'top_topics', 'topic_counts']
df_final = df_final[cols]


In [ ]:
df_final

,sector,sentiment,jumlah_ulasan,topik_dan_jumlah,top_topics,topic_counts
9,ALL SECTOR,Positif,170,"membantu (5), tempatnya luas (3), pelayananany...","membantu, tempatnya luas, pelayanananya tempat...","5, 3, 2, 1, 1"
8,ALL SECTOR,Netral,31,"pulkam nugas kuliah (1), nugas kuliah disini (...","pulkam nugas kuliah, nugas kuliah disini, pulk...","1, 1, 1, 1, 1"
7,ALL SECTOR,Negatif,26,"pelayanannya minus tolong (4), reguler pelayan...","pelayanannya minus tolong, reguler pelayananny...","4, 4, 4, 1, 1"
2,Harbourfront and Intermoda Area,Positif,105,"membantu (5), tempatnya luas (3), pelayananany...","membantu, tempatnya luas, pelayanananya tempat...","5, 3, 2, 1, 1"
1,Harbourfront and Intermoda Area,Netral,23,"pulkam nugas kuliah (1), nugas kuliah disini (...","pulkam nugas kuliah, nugas kuliah disini, pulk...","1, 1, 1, 1, 1"
0,Harbourfront and Intermoda Area,Negatif,21,"pelayanannya minus tolong (4), reguler pelayan...","pelayanannya minus tolong, reguler pelayananny...","4, 4, 4, 1, 1"
5,Hilltop Resort,Positif,53,"parkir jalan utama (1), mobil parkir jalan (1)...","parkir jalan utama, mobil parkir jalan, parkir...","1, 1, 1, 1, 1"
4,Hilltop Resort,Netral,8,"membeli berbagai kebutuhan (1), perjalanan mem...","membeli berbagai kebutuhan, perjalanan membeli...","1, 1, 1, 1, 1"
3,Hilltop Resort,Negatif,5,"ramai perbekalan penyebrangan (1), berdatangan...","ramai perbekalan penyebrangan, berdatangan ram...","1, 1, 1, 1, 1"
6,Marina District,Positif,12,"gurme bakar nya (1), ayam bakar bekakak (1), b...","gurme bakar nya, ayam bakar bekakak, bakar nya...","1, 1, 1, 1, 1"


In [ ]:
# ubah struktur
import json

data = df_final.to_dict(orient='records')

structured_data = {}

for item in data:
    sec = item['sector']

    if sec not in structured_data:
        structured_data[sec] = {
            "sector": sec,
            "total_ulasan": 0,
            "all_topik_dan_jumlah_list": [],
            "sentiment_breakdown": []
        }

    jml = int(item['jumlah_ulasan']) if item['jumlah_ulasan'] else 0
    structured_data[sec]["total_ulasan"] += jml

    if item['topik_dan_jumlah']:
        structured_data[sec]["all_topik_dan_jumlah_list"].append(str(item['topik_dan_jumlah']))

    structured_data[sec]["sentiment_breakdown"].append({
        "sentiment": item['sentiment'],
        "jumlah_ulasan": jml,
        "top_topics": item['top_topics'],
        "topic_counts": item['topic_counts']
    })

final_output = []
for sec_data in structured_data.values():
    final_output.append({
        "sector": sec_data["sector"],
        "total_ulasan": sec_data["total_ulasan"],
        "all_topik_dan_jumlah": ", ".join(sec_data["all_topik_dan_jumlah_list"]),
        "sentiment_breakdown": sec_data["sentiment_breakdown"]
    })

with open('1final_summary.json', 'w') as f:
    json.dump(final_output, f, indent=4)

In [ ]:
pd.read_json('1final_summary.json')

,sector,total_ulasan,all_topik_dan_jumlah,sentiment_breakdown
0,ALL SECTOR,227,"membantu (5), tempatnya luas (3), pelayananany...","[{'sentiment': 'Positif', 'jumlah_ulasan': 170..."
1,Harbourfront and Intermoda Area,149,"membantu (5), tempatnya luas (3), pelayananany...","[{'sentiment': 'Positif', 'jumlah_ulasan': 105..."
2,Hilltop Resort,66,"parkir jalan utama (1), mobil parkir jalan (1)...","[{'sentiment': 'Positif', 'jumlah_ulasan': 53,..."
3,Marina District,12,"gurme bakar nya (1), ayam bakar bekakak (1), b...","[{'sentiment': 'Positif', 'jumlah_ulasan': 12,..."


## rekomendasi AI

In [ ]:
!pip install groq

import os
import pandas as pd
from groq import Groq
from google.colab import userdata

# Inisialisasi Client Groq menggunakan Secrets Colab
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 6.1 MB/s eta 0:00:00


### per sektor

In [ ]:
def generate_rekomendasi_groq(sector_name, total, topics, breakdown):
    # Prompt tetap sama karena instruksinya sudah standar profesional
    prompt = f"""
    Anda adalah konsultan bisnis profesional.
    Berdasarkan data berikut, berikan 2-3 rekomendasi strategis singkat untuk sektor ini.

    Nama Sektor: {sector_name}
    Total Ulasan: {total}
    Topik Utama: {topics}
    Detail Sentimen: {breakdown}

    Berdasarkan data di atas, buatlah satu paragraf kesimpulan (maksimal 150 kata) yang mencakup:
    Analisis masalah utama (dari sisi negatif/netral), peluang strategis yang bisa dikembangkan (dari sisi positif), serta rekomendasi tindakan konkret untuk meningkatkan kepuasan pengunjung.

    Gunakan Bahasa Indonesia profesional, lugas, dan pastikan hasilnya hanya terdiri dari SATU paragraf saja.
    """

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model="llama-3.3-70b-versatile",
        temperature=0.5,
    )
    return chat_completion.choices[0].message.content

In [ ]:
for item in final_output:
    print(f"Mengolah Rekomendasi Groq untuk: {item['sector']}...")

    # Memasukkan argumen yang sesuai ke fungsi
    rekomendasi = generate_rekomendasi_groq(
        item['sector'],
        item['total_ulasan'],
        item['all_topik_dan_jumlah'],
        json.dumps(item['sentiment_breakdown'], indent=2) # Pakai json.dumps lebih rapi
    )

    item['rekomendasi_ai'] = rekomendasi

# Simpan ke JSON baru
with open('1final_summary_with_recommend.json', 'w') as f:
    json.dump(final_output, f, indent=4)

print("Selesai! File '1final_summary_with_recommend.json' telah dibuat.")

Mengolah Rekomendasi Groq untuk: ALL SECTOR...
Mengolah Rekomendasi Groq untuk: Harbourfront and Intermoda Area...
Mengolah Rekomendasi Groq untuk: Hilltop Resort...
Mengolah Rekomendasi Groq untuk: Marina District...
Selesai! File '1final_summary_with_recommend.json' telah dibuat.


In [ ]:
pd.read_json('1final_summary_with_recommend.json')

,sector,total_ulasan,all_topik_dan_jumlah,sentiment_breakdown,rekomendasi_ai
0,ALL SECTOR,227,"membantu (5), tempatnya luas (3), pelayananany...","[{'sentiment': 'Positif', 'jumlah_ulasan': 170...","Berdasarkan data yang disajikan, terdapat bebe..."
1,Harbourfront and Intermoda Area,149,"membantu (5), tempatnya luas (3), pelayananany...","[{'sentiment': 'Positif', 'jumlah_ulasan': 105...","Berdasarkan data ulasan, terdapat masalah utam..."
2,Hilltop Resort,66,"parkir jalan utama (1), mobil parkir jalan (1)...","[{'sentiment': 'Positif', 'jumlah_ulasan': 53,...","Berdasarkan data ulasan Hilltop Resort, terdap..."
3,Marina District,12,"gurme bakar nya (1), ayam bakar bekakak (1), b...","[{'sentiment': 'Positif', 'jumlah_ulasan': 12,...","Berdasarkan data ulasan Marina District, terli..."


### per sentimen

In [ ]:
def get_final_analysis(row):
    """Fungsi untuk memproses rekomendasi per baris"""
    # Ambil data dari kolom yang tersedia di row
    sector = row['sector']
    sentiment = row['sentiment']

    topics = row['topik_dan_jumlah']

    prompt = f"""
    Sektor: {sector}
    Sentimen: {sentiment}
    Topik Utama: {topics}

    Tugas: Berikan 1 rekomendasi bisnis singkat (maks 1 paragraf) yang spesifik untuk manajemen.
    Jika sentimen positif, berikan saran apresiasi/promosi.
    Jika negatif, berikan solusi perbaikan konkret.
    Bahasa: Indonesia yang profesional.
    """

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": "Anda adalah konsultan strategis pariwisata."},
                {"role": "user", "content": prompt}
            ],
            model="llama-3.3-70b-versatile",
            temperature=0.3,
        )
        return chat_completion.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

# Jalankan ulang apply-nya
df_final['rekomendasi_ai'] = df_final.apply(get_final_analysis, axis=1)

print("Analisis Selesai!")
display(df_final[['sector', 'sentiment', 'topik_dan_jumlah', 'rekomendasi_ai']])

Analisis Selesai!


,sector,sentiment,topik_dan_jumlah,rekomendasi_ai
9,ALL SECTOR,Positif,"membantu (5), tempatnya bersih (4), semoga suk...","Dengan sentimen positif yang kuat, saya mereko..."
8,ALL SECTOR,Netral,"pulau jawa (2), pulkam nugas (1), nugas kuliah...",Berdasarkan topik utama yang terkait dengan pa...
7,ALL SECTOR,Negatif,"pelayanannya minus (4), reguler pelayanannya (...","Berdasarkan sentimen negatif yang diberikan, k..."
2,Harbourfront and Intermoda Area,Positif,"membantu (5), tempatnya bersih (3), dermaga ek...",Berdasarkan sentimen positif yang diterima dar...
1,Harbourfront and Intermoda Area,Netral,"pulau jawa (2), pulkam nugas (1), nugas kuliah...",Berdasarkan analisis sektor Harbourfront and I...
0,Harbourfront and Intermoda Area,Negatif,"pelayanannya minus (4), reguler pelayanannya (...","Berdasarkan sentimen negatif yang diterima, ka..."
5,Hilltop Resort,Positif,"masjid bsi (3), view nya (2), indah view (2), ...",Berdasarkan sentimen positif dari pengunjung H...
4,Hilltop Resort,Netral,"membeli berbagai (1), berbagai kebutuhan (1), ...",Berdasarkan analisis sentimen netral terkait H...
3,Hilltop Resort,Negatif,"ramai perbekalan (1), kulkas rusak (1), dekat ...","Berdasarkan sentimen negatif yang disampaikan,..."
6,Marina District,Positif,"semoga sukses (4), sukses (4), semoga (4), bak...",Berdasarkan analisis sentimen positif di Sekto...


In [ ]:
df_final

,sector,sentiment,jumlah_ulasan,topik_dan_jumlah,rekomendasi_ai
9,ALL SECTOR,Positif,181,"membantu (5), tempatnya bersih (4), semoga suk...","Dengan sentimen positif yang kuat, saya mereko..."
8,ALL SECTOR,Netral,31,"pulau jawa (2), pulkam nugas (1), nugas kuliah...",Berdasarkan topik utama yang terkait dengan pa...
7,ALL SECTOR,Negatif,35,"pelayanannya minus (4), reguler pelayanannya (...","Berdasarkan sentimen negatif yang diberikan, k..."
2,Harbourfront and Intermoda Area,Positif,113,"membantu (5), tempatnya bersih (3), dermaga ek...",Berdasarkan sentimen positif yang diterima dar...
1,Harbourfront and Intermoda Area,Netral,23,"pulau jawa (2), pulkam nugas (1), nugas kuliah...",Berdasarkan analisis sektor Harbourfront and I...
0,Harbourfront and Intermoda Area,Negatif,29,"pelayanannya minus (4), reguler pelayanannya (...","Berdasarkan sentimen negatif yang diterima, ka..."
5,Hilltop Resort,Positif,56,"masjid bsi (3), view nya (2), indah view (2), ...",Berdasarkan sentimen positif dari pengunjung H...
4,Hilltop Resort,Netral,8,"membeli berbagai (1), berbagai kebutuhan (1), ...",Berdasarkan analisis sentimen netral terkait H...
3,Hilltop Resort,Negatif,6,"ramai perbekalan (1), kulkas rusak (1), dekat ...","Berdasarkan sentimen negatif yang disampaikan,..."
6,Marina District,Positif,12,"semoga sukses (4), sukses (4), semoga (4), bak...",Berdasarkan analisis sentimen positif di Sekto...
